# Libraries

In [ ]:
import gc
import csv
import time
import yaml
import shutil
import random
import kagglehub
import numpy as np
import pandas as pd
from tqdm import tqdm
from PIL import Image
from pathlib import Path
import matplotlib.pyplot as plt 
from collections import Counter

import torch
import torch.nn as nn
import torch.optim as optim
import torchvision.utils as vutils
import torchvision.transforms as T
from torch.utils.data import Dataset, DataLoader

# Utils

### DOWNLOAD=True if you want to import into Input in Kaggle Notebook and download in Google Colab

CHANGE `DATASET_PATH` TO APPROPRIATE PATH IF ON GOOGLE COLAB

https://www.kaggle.com/datasets/dimensi0n/imagenet-256

In [ ]:
DOWNLOAD = False

#### Insert your username below

In [ ]:
username = ""
MODEL_PATH = Path(f"/kaggle/input/models/{username}/mnv4/pytorch/default/1")
DATASET_PATH = f"/kaggle/input/datasets/{username}/imagenet/imagenet"

In [ ]:
def set_seed(seed: int = 42, deterministic: bool = False):
    random.seed(seed)                     # Python random
    np.random.seed(seed)                  # NumPy
    torch.manual_seed(seed)               # CPU
    torch.cuda.manual_seed(seed)          # GPU
    torch.cuda.manual_seed_all(seed)      # All GPUs
    torch.backends.cudnn.deterministic = deterministic  # Deterministic convs
    torch.backends.cudnn.benchmark = not deterministic  # Auto-tune conv algorithms when not in deterministic mode
    print(f"Random seed set to {seed}")

def save_training_plots(
    model_name,
    loss_history,
    train_acc_history,
    test_acc_history,
    epoch_times,
    output_dir="outputs/plots"
):
    output_dir = Path(output_dir)
    output_dir.mkdir(parents=True, exist_ok=True)
    epochs = np.arange(1, len(loss_history) + 1)

    # Loss plot
    plt.figure()
    plt.plot(epochs, loss_history, label="Train Loss")
    plt.xlabel("Epoch")
    plt.ylabel("Loss")
    plt.title(f"Training Loss - {model_name}")
    plt.legend()
    plt.grid(True)
    plt.savefig(output_dir / f"{model_name}_loss.png")
    plt.close()

    # Accuracy plot
    plt.figure()
    plt.plot(epochs, train_acc_history, label="Train Accuracy")
    plt.plot(epochs, test_acc_history, label="Test Accuracy")
    plt.xlabel("Epoch")
    plt.ylabel("Accuracy")
    plt.title(f"Train vs Test Accuracy - {model_name}")
    plt.legend()
    plt.grid(True)
    plt.savefig(output_dir / f"{model_name}_accuracy.png")
    plt.close()

    # Time per epoch plot
    plt.figure()
    plt.plot(epochs, epoch_times, label="Time per Epoch (s)")
    plt.xlabel("Epoch")
    plt.ylabel("Seconds")
    plt.title(f"Epoch Time - {model_name}")
    plt.legend()
    plt.grid(True)
    plt.savefig(output_dir / f"{model_name}_epoch_time.png")
    plt.close()

    print(f"\nPlots saved to: {output_dir.resolve()}")

def summarize_checkpoint_times(ckpt_path):
    ckpt = torch.load(Path("outputs/checkpoints/") / ckpt_path, map_location="cpu")
    
    # Check if epoch_times exists
    if "epoch_times" not in ckpt:
        print("Checkpoint does not contain 'epoch_times'.")
        return None

    epoch_times = ckpt["epoch_times"]
    total_time = sum(epoch_times)
    avg_time = total_time / len(epoch_times)

    def format_hms(seconds):
        h = int(seconds // 3600)
        m = int((seconds % 3600) // 60)
        s = int(seconds % 60)
        return f"{h}h {m}m {s}s"

    print(f"Average epoch time: {format_hms(avg_time)}")
    print(f"Total training time: {format_hms(total_time)}")
    
    return avg_time, total_time

# Config

In [ ]:
DATA_CFG = {
    "root": "data/imagenet",
    "image_size": 224,

    "mean": [0.485, 0.456, 0.406],
    "std": [0.229, 0.224, 0.225],

    "batch_size": 256,
    "num_workers": 4,

    "split_ratio": 0.8,
    "num_classes": 100,
}

MOBILENET_CONFIG = {
    "optimizer": "adamw",
    "lr": 4e-3,
    "weight_decay": 0.01,
    "label_smoothing": 0.1,
    "epochs": 100,
    "warmup_epochs": 5,
    "start_from": None,
    "batch_size": 256,
}

RESNET_CONFIG = {
    "optimizer": "sgd",
    "lr": 0.1,
    "momentum": 0.9,
    "weight_decay": 1e-4,
    "label_smoothing": 0.0,
    "epochs": 100,
    "warmup_epochs": 5,
    "start_from": None,
    "batch_size": 128,
}

DENSENET_CONFIG = {
    "optimizer": "sgd",
    "lr": 0.1,
    "momentum": 0.9,
    "weight_decay": 1e-4,
    "label_smoothing": 0.0,
    "epochs": 100,
    "warmup_epochs": 5,
    "start_from": None,
    "batch_size": 96,
}

# Dataset

In [ ]:
def download_data(data_dir):
    data_dir = Path(data_dir)
    data_dir.mkdir(parents=True, exist_ok=True)

    download_path = kagglehub.dataset_download("dimensi0n/imagenet-256")

    print("Path to dataset files:", download_path)

    print(f"Moving data into {data_dir} ...")
    for content_path in Path(download_path).iterdir():
        shutil.move(content_path, data_dir)
    print("Moving complete!")
    return data_dir

In [ ]:
if DOWNLOAD:
    download_data(DATA_CFG['root'])

In [ ]:
def process_data(download):
    if download:
        data_path = download_data(DATASET_PATH)
    else:
        data_path = DATASET_PATH

    data_path = Path(data_path)
    train_path = Path(data_path) / "train"
    test_path = Path(data_path) / "test"
    print(f"{train_path}")
    print(f"{test_path}")

    train_path.mkdir(parents=True, exist_ok=True)
    test_path.mkdir(parents=True, exist_ok=True)

    for class_dir in tqdm(list(data_path.iterdir())):
        print()
        if class_dir == train_path or class_dir == test_path: continue
        if class_dir.is_dir():
            img_paths = []
            class_name = class_dir.name
            
            for img_path in class_dir.iterdir():
                if img_path.suffix.lower() in [".jpg", ".jpeg", ".png"]:
                    img_paths.append(img_path)

            random.shuffle(img_paths)

            split_idx = int(len(img_paths) * DATA_CFG["split_ratio"])
            train_imgs = img_paths[:split_idx]
            test_imgs   = img_paths[split_idx:]

            (train_path / class_name).mkdir(parents=True, exist_ok=True)
            (test_path / class_name).mkdir(parents=True, exist_ok=True)

            for img in train_imgs:
                shutil.copy(img, train_path / class_name / img.name)

            for img in test_imgs:
                shutil.copy(img, test_path / class_name / img.name)

            shutil.rmtree(class_dir)
            print(f"{class_name}: {len(train_imgs)} train, {len(test_imgs)} val")

def build_transforms(image_size=224, train=True):
    if train:
        return T.Compose([
            T.RandomResizedCrop(image_size),
            T.RandomHorizontalFlip(),
            T.ToTensor(),
            T.Normalize(mean=DATA_CFG["mean"], std=DATA_CFG["std"])
        ])
    else:
        return T.Compose([
            T.Resize(256),
            T.CenterCrop(image_size),
            T.ToTensor(),
            T.Normalize(mean=DATA_CFG["mean"], std=DATA_CFG["std"])
        ])

class ImageNetDataset(Dataset):
    def __init__(self, root, split="train", transform=None):
        self.root = Path(root) / split
        self.transform = transform

        # Scan class folders
        all_dirs = sorted(d.name for d in self.root.iterdir() if d.is_dir())
        n = DATA_CFG.get("num_classes", 1000)
        self.classes = all_dirs[::max(1, len(all_dirs) // n)][:n]

        self.class_to_idx = {cls_name: i for i, cls_name in enumerate(self.classes)}
        self.idx_to_class = {i: cls_name for i, cls_name in enumerate(self.classes)}

        # Build list of (image_path, label)
        self.samples = []
        for cls_name in self.classes:
            cls_folder = self.root / cls_name
            for img_path in cls_folder.iterdir():
                if img_path.suffix.lower() in [".jpg", ".jpeg", ".png"]:
                    self.samples.append((img_path, self.class_to_idx[cls_name]))

    def __len__(self):
        return len(self.samples)
    
    def __getitem__(self, index):
        img_path, label = self.samples[index]
        image = Image.open(img_path).convert("RGB")
        if self.transform:
            image = self.transform(image)
        return image, label

# Get dataset mean and standard deviation

In [ ]:
MEAN_STD = False

In [ ]:
if MEAN_STD:
    transform = T.Compose([
        T.ToTensor()
    ])
    
    train_datasets = ImageNetDataset(
            root=DATASET_PATH, 
            transform=transform)
    
    loader = DataLoader(train_datasets, batch_size=128, shuffle=False)
    
    mean = torch.zeros(3)
    std = torch.zeros(3)
    total_pixels = 0
    
    for images, _ in tqdm(loader):
        b, c, h, w = images.shape
        num_pixels = b * h * w
    
        mean += images.sum(dim=[0, 2, 3])
        std += (images ** 2).sum(dim=[0, 2, 3])
        total_pixels += num_pixels
    
    mean /= total_pixels
    std = torch.sqrt(std / total_pixels - mean ** 2)
    
    print("Mean:", mean)
    print("Std:", std)

# Model

### MobileNetV4

In [ ]:
class ConvBNReLU(nn.Sequential):
    def __init__(self, in_channel, out_channel, k=3, stride=1, groups=1, activation=True):
        layers = [
            nn.Conv2d(in_channel, out_channel, k, stride, padding=k//2,
                      groups=groups, bias=False),
            nn.BatchNorm2d(out_channel)
        ]
        if activation:
            layers.append(nn.ReLU(inplace=True))
            
        super(ConvBNReLU, self).__init__(*layers)

In [ ]:
class FusedIB(nn.Module):
    def __init__(self, in_channel, exp_channel, out_channel, stride):
        super(FusedIB, self).__init__()
        self.expansion = ConvBNReLU(in_channel, exp_channel, k=3, stride=stride)
        self.project = ConvBNReLU(exp_channel, out_channel, k=1, activation=False)
        self.use_skip = stride == 1 and in_channel == out_channel

    def forward(self, x):
        out = self.project(self.expansion(x))
        return out + x if self.use_skip else out

In [ ]:
class UIB(nn.Module):
    def __init__(self, in_channel, exp_channel, out_channel, k1=0, k2=0, stride=1):
        super(UIB, self).__init__()

        # stride goes on middle DW if present, else start DW
        s1 = stride if (k1 and not k2) else 1
        s2 = stride if k2 else 1

        self.start_dw = (ConvBNReLU(in_channel, in_channel, k=k1, stride=s1,
                                    groups=in_channel, activation=False)
                         if k1 else nn.Identity())
        self.expand = ConvBNReLU(in_channel, exp_channel, k=1)
        self.middle_dw = (ConvBNReLU(exp_channel, exp_channel, k=k2, stride=s2,
                                     groups=exp_channel)
                          if k2 else nn.Identity())
        self.project = ConvBNReLU(exp_channel, out_channel, k=1, activation=False)

        self.use_skip = stride == 1 and in_channel == out_channel

    def forward(self, x):
        out = self.start_dw(x)
        out = self.expand(out)
        out = self.middle_dw(out)
        out = self.project(out)
        return out + x if self.use_skip else out

In [ ]:
class MobileNetV4ConvS(nn.Module):
    def __init__(self, num_classes=1000, dropout=0.3):
        super(MobileNetV4ConvS, self).__init__()
        self.UIB_CFG = [
            (5, 5, 192,  96, 2),
            (0, 3, 192,  96, 1),
            (0, 3, 192,  96, 1),
            (0, 3, 192,  96, 1),
            (0, 3, 192,  96, 1),
            (3, 0, 384,  96, 1),
            (3, 3, 576, 128, 2),
            (5, 5, 512, 128, 1),
            (0, 5, 512, 128, 1),
            (0, 5, 384, 128, 1),
            (0, 3, 512, 128, 1),
            (0, 3, 512, 128, 1),
        ]

        layers = [
            ConvBNReLU(3, 32, k=3, stride=2),
            FusedIB(32, 32, 32, stride=2),
            FusedIB(32, 96, 64, stride=2),
        ]

        in_channel = 64
        for k1, k2, exp_channel, out_channel, stride in self.UIB_CFG:
            layers.append(UIB(in_channel, exp_channel, out_channel, k1, k2, stride))
            in_channel = out_channel

        layers.append(ConvBNReLU(in_channel, 960, k=1))
        self.features = nn.Sequential(*layers)

        self.head = nn.Sequential(
            nn.AdaptiveAvgPool2d(1),
            nn.Conv2d(960, 1280, kernel_size=1),
            nn.ReLU(inplace=True),
            nn.Dropout(dropout),
            nn.Conv2d(1280, num_classes, kernel_size=1),
        )   

        self._init_weights()

    def forward(self, x):
        x = self.features(x)
        x = self.head(x)
        return x.flatten(1)

    def _init_weights(self):
        for m in self.modules():
            if isinstance(m, nn.Conv2d):
                nn.init.kaiming_normal_(m.weight, mode="fan_out", nonlinearity="relu")
                if m.bias is not None:
                    nn.init.zeros_(m.bias)
            elif isinstance(m, nn.BatchNorm2d):
                nn.init.ones_(m.weight)
                nn.init.zeros_(m.bias)

### ResNet

In [ ]:
class block(nn.Module):
    def __init__(self, in_channels, out_channels, identity_downsample=None, stride=1):
        super(block, self).__init__()
        self.expansion = 4
        self.conv1 = nn.Conv2d(in_channels, out_channels, kernel_size=1, stride=1, padding=0, bias=False)
        self.bn1 = nn.BatchNorm2d(out_channels)
        self.conv2 = nn.Conv2d(out_channels, out_channels, kernel_size=3, stride=stride, padding=1, bias=False)
        self.bn2 = nn.BatchNorm2d(out_channels)
        self.conv3 = nn.Conv2d(out_channels, out_channels*self.expansion, kernel_size=1, stride=1, padding=0, bias=False)
        self.bn3 = nn.BatchNorm2d(out_channels*self.expansion)
        self.relu = nn.ReLU(inplace=True)
        self.identity_downsample = identity_downsample

    def forward(self, x):
        identity = x

        x = self.conv1(x)
        x = self.bn1(x)
        x = self.relu(x)
        x = self.conv2(x)
        x = self.bn2(x)
        x = self.relu(x)
        x = self.conv3(x)
        x = self.bn3(x)

        if self.identity_downsample is not None:
            identity = self.identity_downsample(identity)
        
        x = x + identity
        x = self.relu(x)
        return x
    
class ResNet(nn.Module):
    def __init__(self, block, layers, image_channels, num_classes):
        super(ResNet, self).__init__()
        self.in_channels = 64
        self.conv1 = nn.Conv2d(image_channels, 64, kernel_size=7, stride=2, padding=3, bias=False)
        self.bn1 = nn.BatchNorm2d(64)
        self.relu = nn.ReLU(inplace=True)
        self.maxpool = nn.MaxPool2d(kernel_size=3, stride=2, padding=1)

        # ResNet Layers
        self.layer1 = self._make_layer(block, layers[0], out_channels=64, stride=1)
        self.layer2 = self._make_layer(block, layers[1], out_channels=128, stride=2)
        self.layer3 = self._make_layer(block, layers[2], out_channels=256, stride=2)
        self.layer4 = self._make_layer(block, layers[3], out_channels=512, stride=2)

        self.avgpool = nn.AdaptiveAvgPool2d((1,1))
        self.fc = nn.Linear(512*4, num_classes)

        self._init_weights()

    def forward(self, x):
        x = self.conv1(x)
        x = self.bn1(x)
        x = self.relu(x)
        x = self.maxpool(x)

        x = self.layer1(x)
        x = self.layer2(x)
        x = self.layer3(x)
        x = self.layer4(x)

        x = self.avgpool(x)
        x = x.reshape(x.shape[0], -1)
        x = self.fc(x)

        return x

    def _make_layer(self, block, num_residual_blocks, out_channels, stride):
        identity_downsample = None
        layers = []

        if stride != 1 or self.in_channels != out_channels * 4:
            identity_downsample = nn.Sequential(nn.Conv2d(self.in_channels, out_channels*4, kernel_size=1,
                                                          stride=stride, bias=False),
                                                nn.BatchNorm2d(out_channels*4))
        
        layers.append(block(self.in_channels, out_channels, identity_downsample, stride))
        self.in_channels = out_channels * 4

        for i in range(num_residual_blocks - 1):
            layers.append(block(self.in_channels, out_channels))
        
        return nn.Sequential(*layers)

    def _init_weights(self):
        for m in self.modules():
            if isinstance(m, nn.Conv2d):
                nn.init.kaiming_normal_(m.weight, mode="fan_out", nonlinearity="relu")
            elif isinstance(m, nn.BatchNorm2d):
                nn.init.ones_(m.weight)
                nn.init.zeros_(m.bias)
    
def ResNet50(img_channels=3, num_classes=100):
    return ResNet(block, [3, 4, 6, 3], img_channels, num_classes)
    
def ResNet101(img_channels=3, num_classes=100):
    return ResNet(block, [3, 4, 23, 3], img_channels, num_classes)
    
def ResNet152(img_channels=3, num_classes=100):
    return ResNet(block, [3, 8, 36, 3], img_channels, num_classes)

In [ ]:
model = ResNet50(num_classes=100)
print(sum(p.numel() for p in model.parameters()) / 1e6)
print(model(torch.randn(2, 3, 224, 224)).shape)

### DenseNet

In [ ]:
class DenseLayer(nn.Module):
    def __init__(self, in_channels, growth_rate=32):
        super(DenseLayer, self).__init__()
        self.bn1 = nn.BatchNorm2d(in_channels)
        self.relu = nn.ReLU(inplace=True)
        self.conv1 = nn.Conv2d(in_channels, out_channels=4*growth_rate, kernel_size=1, bias=False)
        self.bn2 = nn.BatchNorm2d(4*growth_rate)
        self.conv2 = nn.Conv2d(4*growth_rate, out_channels=growth_rate, kernel_size=3, stride=1, padding=1, bias=False)
    
    def forward(self, x):
        identity = x

        x = self.bn1(x)
        x = self.relu(x)
        x = self.conv1(x)
        x = self.bn2(x)
        x = self.relu(x)
        x = self.conv2(x)

        out = torch.cat((identity, x), dim=1)
        return out

class DenseBlock(nn.Module):
    def __init__(self, num_layers, in_channels, growth_rate=32):
        super(DenseBlock, self).__init__()
        self.num_layers = num_layers
        self.in_channels = in_channels
        self.growth_rate = growth_rate

        layers = []
        for _ in range(num_layers):
            layers.append(DenseLayer(in_channels, self.growth_rate))
            in_channels += self.growth_rate

        self.dense_block = nn.Sequential(*layers)
    
    def forward(self, x):
        return self.dense_block(x)

class TransitionLayer(nn.Module):
    def __init__(self, in_channels, compression_factor=0.5):
        super(TransitionLayer, self).__init__()
        out_channels = int(in_channels * compression_factor)

        self.bn = nn.BatchNorm2d(in_channels)
        self.relu = nn.ReLU(inplace=True)
        self.conv = nn.Conv2d(in_channels, out_channels, kernel_size=1, stride=1, bias=False)
        self.pool = nn.AvgPool2d(kernel_size=2, stride=2, padding=0)
    
    def forward(self, x):
        x = self.bn(x)
        x = self.relu(x)
        x = self.conv(x)
        x = self.pool(x)
        return x
    
class DenseNet(nn.Module):
    def __init__(self, image_channels, layers, growth_rate=32, compression_factor=0.5, num_classes=100):
        super(DenseNet, self).__init__()
        self.features = []
        self.features.append(
            nn.Conv2d(image_channels, 64, kernel_size=7, stride=2, padding=3, bias=False)
        )
        self.features.append(nn.BatchNorm2d(64))
        self.features.append(nn.ReLU(inplace=True))
        self.features.append(nn.MaxPool2d(kernel_size=3, stride=2, padding=1))

        num_channels = 64
        for i, num_layers in enumerate(layers):
            dense_block = DenseBlock(num_layers, num_channels, growth_rate=growth_rate)
            self.features.append(dense_block)
            num_channels += num_layers * growth_rate

            if i != len(layers) - 1:
                transition_layer = TransitionLayer(num_channels, compression_factor=compression_factor)
                self.features.append(transition_layer)
                num_channels = int(num_channels * compression_factor)
        
        self.features.append(nn.BatchNorm2d(num_channels))
        self.features.append(nn.ReLU(inplace=True))
        self.features.append(nn.AdaptiveAvgPool2d((1,1)))
        self.features = nn.Sequential(*self.features)
        self.classifier = nn.Linear(num_channels, num_classes)

        self._init_weights()

    def forward(self, x):
        x = self.features(x)
        x = torch.flatten(x, 1)
        x = self.classifier(x)
        return x

    def _init_weights(self):
        for m in self.modules():
            if isinstance(m, nn.Conv2d):
                nn.init.kaiming_normal_(m.weight, mode="fan_out", nonlinearity="relu")
            elif isinstance(m, nn.BatchNorm2d):
                nn.init.ones_(m.weight)
                nn.init.zeros_(m.bias)

def DenseNet121(img_channels=3, num_classes=100):
    return DenseNet(image_channels=img_channels, layers=[6, 12, 24, 16], growth_rate=32, compression_factor=0.5, num_classes=num_classes)

def DenseNet169(img_channels=3, num_classes=100):
    return DenseNet(image_channels=img_channels, layers=[6, 12, 32, 32], growth_rate=32, compression_factor=0.5, num_classes=num_classes)

def DenseNet201(img_channels=3, num_classes=100):
    return DenseNet(image_channels=img_channels, layers=[6, 12, 48, 32], growth_rate=32, compression_factor=0.5, num_classes=num_classes)

def DenseNet264(img_channels=3, num_classes=100):
    return DenseNet(image_channels=img_channels, layers=[6, 12, 64, 48], growth_rate=32, compression_factor=0.5, num_classes=num_classes)

In [ ]:
model = DenseNet121(num_classes=100)
print(sum(p.numel() for p in model.parameters()) / 1e6)
print(model(torch.randn(2, 3, 224, 224)).shape)           # [2, 100]

# Train

In [ ]:
MODEL_REGISTRY = {
    "conv-s":      (lambda nc: MobileNetV4ConvS(num_classes=nc),       MOBILENET_CONFIG),
    "resnet50":    (lambda nc: ResNet50(num_classes=nc),               RESNET_CONFIG),
    "resnet101":   (lambda nc: ResNet101(num_classes=nc),              RESNET_CONFIG),
    "resnet152":   (lambda nc: ResNet152(num_classes=nc),              RESNET_CONFIG),
    "densenet121": (lambda nc: DenseNet121(num_classes=nc),            DENSENET_CONFIG),
    "densenet169": (lambda nc: DenseNet169(num_classes=nc),            DENSENET_CONFIG),
    "densenet201": (lambda nc: DenseNet201(num_classes=nc),            DENSENET_CONFIG),
}

def build_optimizer(model, cfg):
    name = cfg.get("optimizer", "adamw").lower()
    if name == "adamw":
        return optim.AdamW(
            model.parameters(),
            lr=float(cfg.get("lr", 2e-3)),
            weight_decay=float(cfg.get("weight_decay", 0.01)),
        )
    if name == "sgd":
        return optim.SGD(
            model.parameters(),
            lr=float(cfg.get("lr", 0.1)),
            momentum=float(cfg.get("momentum", 0.9)),
            weight_decay=float(cfg.get("weight_decay", 1e-4)),
            nesterov=True,
        )
    raise ValueError(f"Unknown optimizer: {name}")

def train(model_name):
    """
    Train Mobile Network on ImageNet dataset.

    Args:
        model_name (str): "conv-s"
    """
    # Config
    set_seed(42)

    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    print("Using device:", device)

    if model_name not in MODEL_REGISTRY:
        raise ValueError(
            f"Unknown model: {model_name}. Options: {list(MODEL_REGISTRY)}")
    build_fn, MODEL_CFG = MODEL_REGISTRY[model_name]

    # Datasets & loaders
    train_dataset = ImageNetDataset(
        root=DATASET_PATH, 
        split="train", 
        transform=build_transforms(DATA_CFG["image_size"], train=True))
    test_dataset = ImageNetDataset(
        root=DATASET_PATH, 
        split="test", 
        transform=build_transforms(DATA_CFG["image_size"], train=False))

    loader_kwargs = dict(
        batch_size=MODEL_CFG.get("batch_size", DATA_CFG["batch_size"]),
        num_workers=DATA_CFG["num_workers"],
        pin_memory=True,
        persistent_workers=DATA_CFG["num_workers"] > 0,
    )
    train_loader = DataLoader(train_dataset, shuffle=True, drop_last=True, **loader_kwargs)
    test_loader = DataLoader(test_dataset, shuffle=False, drop_last=False, **loader_kwargs)

    # Model, loss, optimizer
    num_classes = DATA_CFG.get("num_classes", 100)
    model = build_fn(num_classes).to(device)
    print(f"{model_name}: {sum(p.numel() for p in model.parameters())/1e6:.2f} M params")

    num_epochs = MODEL_CFG.get("epochs", 100)
    warmup_epochs = int(MODEL_CFG.get("warmup_epochs", 5))

    criterion = nn.CrossEntropyLoss(label_smoothing=float(MODEL_CFG.get("label_smoothing", 0.0)))
    optimizer = build_optimizer(model, MODEL_CFG)
    scheduler = optim.lr_scheduler.SequentialLR(
        optimizer,
        schedulers=[
            optim.lr_scheduler.LinearLR(optimizer, 0.01, 1.0, total_iters=warmup_epochs),
            optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=num_epochs - warmup_epochs),
        ],
        milestones=[warmup_epochs],
    )
    scaler = torch.amp.GradScaler("cuda", enabled=device.type == "cuda")

    # Checkpoint
    output_dir = Path("outputs/checkpoints")
    output_dir.mkdir(parents=True, exist_ok=True)

    start_epoch = 1
    best_acc = 0.0

    loss_history = []
    train_acc_history = []
    test_acc_history = []
    epoch_times = []

    start_from = MODEL_CFG.get("start_from", None)
    if start_from is not None and not isinstance(start_from, str):
        ckpt_path = output_dir / f"{model_name}_last.pth"
        checkpoint = torch.load(ckpt_path, map_location=device)

        model.load_state_dict(checkpoint["model_state"])
        optimizer.load_state_dict(checkpoint["optimizer_state"])
        scheduler.load_state_dict(checkpoint["scheduler_state"])
        if "scaler_state" in checkpoint:
            scaler.load_state_dict(checkpoint["scaler_state"])

        best_acc = checkpoint.get("best_acc", 0.0)

        loss_history = checkpoint.get("loss_history", [])
        train_acc_history = checkpoint.get("train_acc_history", [])
        test_acc_history = checkpoint.get("test_acc_history", [])
        epoch_times = checkpoint.get("epoch_times", [])

        start_epoch = checkpoint["epoch"] + 1

        print(f"Resumed from epoch {start_epoch}")

    # Training loop
    for epoch in range(start_epoch, num_epochs+1):
        start_time = time.time()

        # Training
        model.train()
        running_loss = 0.0
        correct_train = 0
        total_train = 0
        for images, labels in tqdm(train_loader, desc=f"[Train] Epoch {epoch}/{num_epochs}"):
            images = images.to(device, non_blocking=True)
            labels = labels.to(device, non_blocking=True)

            optimizer.zero_grad(set_to_none=True)
            with torch.autocast("cuda", dtype=torch.float16, enabled=device.type == "cuda"):
                outputs = model(images)
                loss = criterion(outputs, labels)

            scaler.scale(loss).backward()
            scaler.step(optimizer)
            scaler.update()

            running_loss += loss.item() * images.size(0)
            correct_train += (outputs.argmax(1) == labels).sum().item()
            total_train += labels.size(0)

        epoch_loss = running_loss / total_train
        train_acc = correct_train / total_train
        loss_history.append(epoch_loss)
        train_acc_history.append(train_acc)

        # Testing
        model.eval()
        correct_test = 0
        total_test = 0
        with torch.no_grad():
            for images, labels in tqdm(test_loader, desc=f"[Test] Epoch {epoch}/{num_epochs}"):
                images = images.to(device, non_blocking=True)
                labels = labels.to(device, non_blocking=True)

                with torch.autocast("cuda", dtype=torch.float16, enabled=device.type == "cuda"):
                    outputs = model(images)

                correct_test += (outputs.argmax(1) == labels).sum().item()
                total_test += labels.size(0)

        test_acc = correct_test / total_test
        test_acc_history.append(test_acc)

        epoch_time = time.time() - start_time
        epoch_times.append(epoch_time)

        print(f"Epoch {epoch} | Loss: {epoch_loss:.4f} | Train Acc: {train_acc*100:.2f}% | "
              f"Test Acc: {test_acc*100:.2f}% | LR: {optimizer.param_groups[0]['lr']:.2e} | "
              f"Time: {epoch_time:.2f}s")
        
        # Save plots
        save_training_plots(
            model_name=model_name,
            loss_history=loss_history,
            train_acc_history=train_acc_history,
            test_acc_history=test_acc_history,
            epoch_times=epoch_times,
            output_dir="outputs/plots"
        )

        is_best = test_acc > best_acc
        if is_best:
            best_acc = test_acc

        # Save checkpoint
        ckpt = {
            "epoch": epoch,
            "model_state": model.state_dict(),
            "optimizer_state": optimizer.state_dict(),
            "scheduler_state": scheduler.state_dict(),
            "scaler_state": scaler.state_dict(),
            "best_acc": best_acc,
            "classes": train_dataset.classes,

            # histories
            "loss_history": loss_history,
            "train_acc_history": train_acc_history,
            "test_acc_history": test_acc_history,
            "epoch_times": epoch_times,
        }

        torch.save(ckpt, output_dir / f"{model_name}_last.pth")
        if is_best:
            torch.save(ckpt, output_dir / f"{model_name}_best.pth")
            print(f"New best: {best_acc*100:.2f}%")

        scheduler.step()
    
    print("\nTraining Summary")
    print(f"Best Test Accuracy: {best_acc*100:.2f}%")
    print(f"Total time: {sum(epoch_times):.2f} seconds")
    print(f"Avg time/epoch: {np.mean(epoch_times):.2f} seconds")
    print(f"Min epoch time: {np.min(epoch_times):.2f} seconds")
    print(f"Max epoch time: {np.max(epoch_times):.2f} seconds")

if __name__ == "__main__":
    model_name = "conv-s"
    train(model_name)

# Inference

In [ ]:
CKPT_DIR = Path(f"/kaggle/input/models/{username}/mobilenetv4/pytorch/default/1")
PLOTS_DIR = Path("outputs/plots")
METRIC_DIR = Path("outputs/metrics")

def build_model(model_name, num_classes, device="cpu"):
    if model_name not in MODEL_REGISTRY:
        raise ValueError(f"Unknown model: {model_name}. Options: {list(MODEL_REGISTRY)}")
    build_fn, _ = MODEL_REGISTRY[model_name]
    return build_fn(num_classes).to(device)


def count_macs(model, input_size=(1, 3, 224, 224), device="cpu"):
    """Multiply-accumulate count for Conv2d and Linear layers."""
    macs = {"total": 0}
    handles = []

    def conv_hook(m, inp, out):
        kernel_macs = (m.in_channels // m.groups) * m.kernel_size[0] * m.kernel_size[1]
        macs["total"] += out.numel() * kernel_macs

    def linear_hook(m, inp, out):
        macs["total"] += out.numel() * m.in_features

    for m in model.modules():
        if isinstance(m, nn.Conv2d):
            handles.append(m.register_forward_hook(conv_hook))
        elif isinstance(m, nn.Linear):
            handles.append(m.register_forward_hook(linear_hook))

    model.eval()
    with torch.no_grad():
        model(torch.randn(*input_size, device=device))
    for h in handles:
        h.remove()

    return macs["total"]


@torch.no_grad()
def benchmark(model, device, image_size=224, batch_sizes=(1, 64), warmup=10, iters=50):
    """Latency and throughput per batch size, plus peak memory on GPU."""
    model.eval()
    results = {}

    for bs in batch_sizes:
        x = torch.randn(bs, 3, image_size, image_size, device=device)

        for _ in range(warmup):
            model(x)
        if device.type == "cuda":
            torch.cuda.synchronize()

        t0 = time.perf_counter()
        for _ in range(iters):
            model(x)
        if device.type == "cuda":
            torch.cuda.synchronize()
        elapsed = time.perf_counter() - t0

        results[bs] = {
            "latency_ms": elapsed / iters * 1000,
            "throughput": bs * iters / elapsed,
        }

    if device.type == "cuda":
        torch.cuda.empty_cache()
        torch.cuda.reset_peak_memory_stats()
        model(torch.randn(batch_sizes[-1], 3, image_size, image_size, device=device))
        torch.cuda.synchronize()
        results["peak_mem_mb"] = torch.cuda.max_memory_allocated() / 1e6

    return results


def cpu_sweep(model_names=None, thread_counts=(1, 4), image_size=None):
    """Batch-1 CPU latency at different thread counts. The roofline experiment."""
    import platform
    names = model_names or ["conv-s", "resnet50", "densenet121"]
    image_size = image_size or DATA_CFG["image_size"]
    num_classes = DATA_CFG.get("num_classes", 100)
    cpu = torch.device("cpu")

    print(f"\nCPU: {platform.processor()}")
    print(f"Cores: {torch.get_num_threads()} default\n")

    rows = []
    for name in names:
        model = build_model(name, num_classes, cpu)
        macs = count_macs(model, (1, 3, image_size, image_size), cpu)
        row = {"model": name, "macs_G": macs / 1e9}

        for nt in thread_counts:
            torch.set_num_threads(nt)
            r = benchmark(model, cpu, image_size, batch_sizes=(1,), warmup=5, iters=20)
            row[f"cpu_{nt}thread_ms"] = r[1]["latency_ms"]
            print(f"{name:<14} {nt} thread(s): {r[1]['latency_ms']:7.1f} ms")

        rows.append(row)
        del model

    torch.set_num_threads(torch.get_num_threads())

    METRIC_DIR.mkdir(parents=True, exist_ok=True)
    out = METRIC_DIR / "cpu_benchmark.csv"
    with open(out, "w", newline="") as f:
        w = csv.DictWriter(f, fieldnames=list(rows[0].keys()))
        w.writeheader()
        w.writerows(rows)
    print(f"\nSaved to: {out}")

    # Ratios relative to resnet50
    base = next((r for r in rows if r["model"] == "resnet50"), None)
    if base:
        print(f"\n{'model':<14}{'MACs ratio':>12}{'CPU 1t speedup':>16}")
        print("-" * 42)
        for r in rows:
            print(f"{r['model']:<14}"
                  f"{base['macs_G']/r['macs_G']:>12.1f}x"
                  f"{base['cpu_1thread_ms']/r['cpu_1thread_ms']:>15.1f}x")

    return rows
    

def profile_model(model_name, num_classes=None, image_size=None, csv_out=True):
    """Params, MACs, GPU and CPU timing. No checkpoint needed."""
    set_seed(42)
    num_classes = num_classes or DATA_CFG.get("num_classes", 100)
    image_size = image_size or DATA_CFG["image_size"]
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

    model = build_model(model_name, num_classes, device)
    params = sum(p.numel() for p in model.parameters())
    macs = count_macs(model, (1, 3, image_size, image_size), device)

    print(f"\n=== {model_name} ===")
    print(f"Params: {params/1e6:.2f} M")
    print(f"MACs:   {macs/1e9:.3f} G  ({2*macs/1e9:.3f} GFLOPs)")

    gpu = benchmark(model, device, image_size) if device.type == "cuda" else None
    if gpu:
        print(f"GPU  bs=1:  {gpu[1]['latency_ms']:7.2f} ms  {gpu[1]['throughput']:8.0f} img/s")
        print(f"GPU  bs=64: {gpu[64]['latency_ms']:7.2f} ms  {gpu[64]['throughput']:8.0f} img/s")
        print(f"Peak mem (bs=64): {gpu['peak_mem_mb']:.0f} MB")

    cpu_model = build_model(model_name, num_classes, torch.device("cpu"))
    cpu = benchmark(cpu_model, torch.device("cpu"), image_size,
                    batch_sizes=(1,), warmup=3, iters=10)
    print(f"CPU  bs=1:  {cpu[1]['latency_ms']:7.2f} ms  {cpu[1]['throughput']:8.1f} img/s")

    if csv_out:
        METRIC_DIR.mkdir(parents=True, exist_ok=True)
        path = METRIC_DIR / "model_profiles.csv"
        new = not path.exists()
        with open(path, "a", newline="") as f:
            w = csv.writer(f)
            if new:
                w.writerow(["model", "params_M", "macs_G", "gpu_bs1_ms", "gpu_bs64_ms",
                            "gpu_bs64_imgs", "peak_mem_MB", "cpu_bs1_ms"])
            w.writerow([
                model_name, f"{params/1e6:.2f}", f"{macs/1e9:.3f}",
                f"{gpu[1]['latency_ms']:.2f}" if gpu else "",
                f"{gpu[64]['latency_ms']:.2f}" if gpu else "",
                f"{gpu[64]['throughput']:.0f}" if gpu else "",
                f"{gpu['peak_mem_mb']:.0f}" if gpu else "",
                f"{cpu[1]['latency_ms']:.2f}",
            ])
        print(f"Appended to {path}")

    result = {"model": model_name, "params": params, "macs": macs, "gpu": gpu, "cpu": cpu}

    del model, cpu_model
    if device.type == "cuda":
        torch.cuda.empty_cache()

    return result


def profile_all(model_names=None):
    names = model_names or ["conv-s", "resnet50", "densenet121"]
    return [profile_model(n) for n in names]


def inference(model_name, ckpt_name=None, ckpt_dir=None, topk=(1, 5)):
    """
    Evaluate a trained checkpoint on the test split.

    Args:
        model_name (str): key in MODEL_REGISTRY
        ckpt_name (str): filename, defaults to "<model_name>_best.pth"
        ckpt_dir (str|Path): where checkpoints live, defaults to outputs/checkpoints
        topk (tuple): which top-k accuracies to report
    """
    set_seed(42)
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    print("Using device:", device)

    ckpt_dir = Path(ckpt_dir) if ckpt_dir else CKPT_DIR
    ckpt_name = ckpt_name or f"{model_name}_best.pth"
    ckpt_path = ckpt_dir / ckpt_name
    if not ckpt_path.exists():
        raise FileNotFoundError(f"No checkpoint at {ckpt_path}")

    plots_dir = PLOTS_DIR / model_name
    plots_dir.mkdir(parents=True, exist_ok=True)
    METRIC_DIR.mkdir(parents=True, exist_ok=True)

    # Data
    test_dataset = ImageNetDataset(
        root=DATASET_PATH,
        split="test",
        transform=build_transforms(DATA_CFG["image_size"], train=False),
    )
    test_loader = DataLoader(
        test_dataset,
        batch_size=DATA_CFG["batch_size"],
        shuffle=False,
        num_workers=DATA_CFG["num_workers"],
        pin_memory=True,
    )
    idx_to_class = test_dataset.classes          # list: label index -> class name

    # Model
    model = build_model(model_name, DATA_CFG.get("num_classes", 100), device)
    checkpoint = torch.load(ckpt_path, map_location=device)

    if "classes" in checkpoint:
        assert checkpoint["classes"] == test_dataset.classes, (
            "Class list mismatch between checkpoint and dataset. "
            "Accuracy would be meaningless.")
    if checkpoint.get("model_name", model_name) != model_name:
        raise ValueError(
            f"Checkpoint is for {checkpoint['model_name']}, not {model_name}")

    model.load_state_dict(checkpoint["model_state"])
    model.eval()

    # Metrics tracking
    total = 0
    topk_correct = [0] * len(topk)
    confusion_counter = Counter()      # (true, pred)
    per_class_total = Counter()        # true
    per_class_correct = Counter()      # true & correct

    with torch.no_grad():
        for images, labels in tqdm(test_loader, desc=f"[Inference] {model_name}"):
            images = images.to(device, non_blocking=True)
            labels = labels.to(device, non_blocking=True)

            with torch.autocast("cuda", dtype=torch.float16, enabled=device.type == "cuda"):
                logits = model(images)

            maxk = max(topk)
            topk_preds = logits.topk(maxk, dim=1).indices    # softmax is unnecessary
            hit = topk_preds == labels.unsqueeze(1)
            for i, k in enumerate(topk):
                topk_correct[i] += hit[:, :k].any(dim=1).sum().item()

            preds = topk_preds[:, 0]
            for t, p in zip(labels.cpu().numpy(), preds.cpu().numpy()):
                per_class_total[t] += 1
                if t == p:
                    per_class_correct[t] += 1
                else:
                    confusion_counter[(t, p)] += 1

            total += labels.size(0)

    accs = {k: topk_correct[i] / total for i, k in enumerate(topk)}
    print(f"\n{model_name} accuracy:")
    for k in topk:
        print(f"Top-{k}: {accs[k]:.4f}")

    # Confusion analysis
    most_confused = confusion_counter.most_common(10)
    print("\nTop 10 most confused class pairs (true -> predicted):")
    for (t, p), count in most_confused:
        print(f"{idx_to_class[t]} -> {idx_to_class[p]} : {count}")

    if most_confused:
        labels_plot = [f"{idx_to_class[t]}->{idx_to_class[p]}" for (t, p), _ in most_confused]
        counts = [c for _, c in most_confused]

        plt.figure(figsize=(10, 5))
        plt.bar(range(len(counts)), counts)
        plt.xticks(range(len(counts)), labels_plot, rotation=45, ha="right")
        plt.ylabel("Count")
        plt.title(f"Top 10 Most Confused Class Pairs - {model_name}")
        plt.tight_layout()
        plt.savefig(plots_dir / f"{model_name}_most_confused_pairs.png", dpi=150)
        plt.close()

        # One row per pair: [true sample | predicted sample]
        by_class = {}
        for img_path, label in test_dataset.samples:
            by_class.setdefault(label, []).append(img_path)

        n = len(most_confused)
        fig, axes = plt.subplots(n, 2, figsize=(7, 3.2 * n))
        axes = axes.reshape(n, 2)
        for idx, ((t, p), count) in enumerate(most_confused):
            for col, cls in enumerate((t, p)):
                ax = axes[idx, col]
                pool = by_class.get(cls, [])
                if pool:
                    ax.imshow(Image.open(random.choice(pool)).convert("RGB"))
                tag = "True" if col == 0 else "Pred"
                ax.set_title(f"{tag}: {idx_to_class[cls]} ({count})", fontsize=9)
                ax.axis("off")

        plt.tight_layout()
        plt.savefig(plots_dir / f"{model_name}_most_confused_pairs_samples.png", dpi=150)
        plt.close()
        print(f"\nConfusion plots saved to: {plots_dir}")

    # Per-class accuracy CSV
    class_accuracy = [
        (cls, idx_to_class[cls], per_class_correct[cls] / per_class_total[cls],
         per_class_correct[cls], per_class_total[cls])
        for cls in per_class_total
    ]
    class_accuracy.sort(key=lambda x: x[2], reverse=True)

    csv_path = METRIC_DIR / f"{model_name}_per_class_accuracy.csv"
    with open(csv_path, "w", newline="") as f:
        w = csv.writer(f)
        w.writerow(["class_id", "class_name", "accuracy", "correct", "total"])
        w.writerows([[c, n_, f"{a:.4f}", ok, tot] for c, n_, a, ok, tot in class_accuracy])
    print(f"Per-class accuracy CSV saved to: {csv_path}")

    # Training curves from the checkpoint histories
    save_training_plots(
        model_name=model_name,
        loss_history=checkpoint.get("loss_history", []),
        train_acc_history=checkpoint.get("train_acc_history", []),
        test_acc_history=checkpoint.get("test_acc_history", []),
        epoch_times=checkpoint.get("epoch_times", []),
        output_dir=plots_dir,
    )

    return {
        "model": model_name,
        "top1": accs.get(1),
        "top5": accs.get(5),
        "epochs": checkpoint.get("epoch"),
        "epoch_times": checkpoint.get("epoch_times", []),
    }


def plot_comparison(rows, out_dir=None):
    """Cross-model figures: accuracy vs cost, and efficiency bars."""
    out_dir = Path(out_dir) if out_dir else PLOTS_DIR
    out_dir.mkdir(parents=True, exist_ok=True)
    valid = [r for r in rows if r.get("top1") is not None]
    if not valid:
        print("No evaluated models, skipping comparison plots.")
        return

    # Accuracy vs params, accuracy vs MACs
    fig, axes = plt.subplots(1, 2, figsize=(12, 5))
    for ax, key, xlabel in [(axes[0], "params_M", "Parameters (M)"),
                            (axes[1], "macs_G", "MACs (G)")]:
        for r in valid:
            ax.scatter(r[key], r["top1"] * 100, s=90)
            ax.annotate(r["model"], (r[key], r["top1"] * 100),
                        textcoords="offset points", xytext=(6, 5), fontsize=9)
        ax.set_xlabel(xlabel)
        ax.set_ylabel("Top-1 accuracy (%)")
        ax.set_xscale("log")
        ax.grid(True, alpha=0.3)
    axes[0].set_title("Accuracy vs model size")
    axes[1].set_title("Accuracy vs compute")
    plt.tight_layout()
    plt.savefig(out_dir / "comparison_accuracy_vs_cost.png", dpi=150)
    plt.close()

    # Efficiency bars
    names = [r["model"] for r in valid]
    metrics = [
        ("params_M", "Params (M)", False),
        ("macs_G", "MACs (G)", False),
        ("cpu_bs1_ms", "CPU latency bs=1 (ms)", False),
        ("gpu_bs64_imgs", "GPU throughput bs=64 (img/s)", True),
    ]
    fig, axes = plt.subplots(1, len(metrics), figsize=(4.5 * len(metrics), 4.5))
    for ax, (key, title, higher_better) in zip(axes, metrics):
        vals = [r[key] if r.get(key) is not None else 0 for r in valid]
        ax.bar(names, vals)
        ax.set_title(title + ("\n(higher better)" if higher_better else "\n(lower better)"),
                     fontsize=10)
        ax.tick_params(axis="x", rotation=30)
        ax.grid(True, axis="y", alpha=0.3)
    plt.tight_layout()
    plt.savefig(out_dir / "comparison_efficiency.png", dpi=150)
    plt.close()
    print(f"Comparison plots saved to: {out_dir}")


def compare(model_names, ckpt_dir=None):
    """Profile and evaluate several models, print one summary table."""
    rows = []
    for name in model_names:
        prof = profile_model(name)
        try:
            ev = inference(name, ckpt_dir=ckpt_dir)
        except FileNotFoundError as e:
            print(f"Skipping eval for {name}: {e}")
            ev = {}

        times = ev.get("epoch_times", [])
        rows.append({
            "model": name,
            "params_M": prof["params"] / 1e6,
            "macs_G": prof["macs"] / 1e9,
            "top1": ev.get("top1"),
            "top5": ev.get("top5"),
            "gpu_bs64_imgs": prof["gpu"][64]["throughput"] if prof["gpu"] else None,
            "cpu_bs1_ms": prof["cpu"][1]["latency_ms"],
            "avg_epoch_s": sum(times) / len(times) if times else None,
        })

    hdr = (f"{'model':<14}{'params(M)':>10}{'MACs(G)':>9}{'top1':>8}{'top5':>8}"
           f"{'GPU img/s':>11}{'CPU ms':>9}{'epoch(s)':>10}")
    print("\n" + hdr)
    print("-" * len(hdr))
    for r in rows:
        def f(v, spec):
            return format(v, spec) if v is not None else "-".rjust(int(spec.strip('>').split('.')[0]))
        print(f"{r['model']:<14}{r['params_M']:>10.2f}{r['macs_G']:>9.3f}"
              f"{f(r['top1'], '>8.4f')}{f(r['top5'], '>8.4f')}"
              f"{f(r['gpu_bs64_imgs'], '>11.0f')}{f(r['cpu_bs1_ms'], '>9.1f')}"
              f"{f(r['avg_epoch_s'], '>10.0f')}")

    METRIC_DIR.mkdir(parents=True, exist_ok=True)
    out = METRIC_DIR / "model_comparison.csv"
    with open(out, "w", newline="") as f:
        w = csv.DictWriter(f, fieldnames=list(rows[0].keys()))
        w.writeheader()
        w.writerows(rows)
    print(f"\nComparison saved to: {out}")

    plot_comparison(rows)
    return rows

if __name__ == "__main__":
    compare(["conv-s", "resnet50", "densenet121"])
    cpu_sweep()